# 👁️ Jev Can See Now: Images + `Box`
### Fast, typed decisions about pictures and screenshots

Jev is TypeSafe AI's System 1 model: you ask typed questions (`Noul`, `Choice`, `Score`) and get calibrated answers back in a single pass.

This notebook runs on **`jev-1.14.0`**, released Oct 14, 2026. At the time of recording, both `jev-latest` and `jev-preview` point to it.

| | `jev-1.13.0` | `jev-1.14.0` | Opt-in header |
|---|---|---|---|
| **`Image`** in the state, instructions or options | ✅ | ✅ | `TypeSafe-Danger-Zone-Images: true` |
| **`Box`**: find something, get where it is | ❌ | ✅ | `TypeSafe-Danger-Zone-Images: true` |
| **Long context**: 262k tokens | ✅ | ✅ | `TypeSafe-Danger-Zone-Long-Context: true` |

TypeSafe says 1.13 "was multimodal all along": images are new to the API, not to the model. `Box` is the new part of 1.14.


# 0. Setup

In [1]:
from pathlib import Path

from dotenv import load_dotenv
from PIL import Image as PILImage, ImageDraw
from typesafe_sdk import TypeSafeClient, Image, Box, Noul, Choice, Score

In [2]:
load_dotenv()

True

---
# 1. Images

## 1.1 The Opt-in Header

Images are behind an opt-in header. Try one without it:

In [3]:
TypeSafeClient().system_one(
    state=Image(Path("images/street.png")),
    questions={"is_japan": Noul(instructions="Was this photo taken in Japan?")},
)

TypeSafeBadRequestError: POST https://api.typesafe.ai/v1/systemone: 400 Images are experimental. Send the header TypeSafe-Danger-Zone-Images: true to use them. (request_id=req_01a175e92f807dc8a59500499a86bee0)

`400 Images are experimental`. The SDK never sends the header for you, so set it once on the client:

In [4]:
client = TypeSafeClient(headers={"TypeSafe-Danger-Zone-Images": "true"})

In [5]:
client.models.list()

ListModelsResponse(models=(ModelMetadata(name='jev-latest', description="The latest iteration of TypeSafe's System One Model: Jev", release_date='2026-09-10T18:38:01.391457+00:00'), ModelMetadata(name='jev-preview', description='A preview version of `jev-latest`: should be better in most ways', release_date='2026-09-10T18:39:06.057655+00:00')))

Two aliases: `jev-latest` (the default) and `jev-preview`. Each response tells you which version actually answered.

## 1.2 An Image as the State

Wrap a file in `Image` and pass it as `state`. Everything from the first video works the same way.

In [ ]:
street = Image(Path("images/street.png"))

In [ ]:
PILImage.open("images/street.png")

In [ ]:
questions = {
    "is_japan": Noul(instructions="Was this photo taken in Japan?"),
    "scene": Choice(
        instructions="What kind of place is this?",
        criteria={
            "shopping_street": "A narrow street lined with shops",
            "highway": "A multi-lane road for fast traffic",
            "park": "Trees, grass, paths",
            "office": "Indoors, desks, computers",
        },
    ),
    "busy": Score(
        instructions="How busy is the street?",
        criteria=["Empty", "Quiet", "Moderate", "Crowded"],
    ),
}

In [ ]:
response = client.system_one(state=street, questions=questions)

In [ ]:
response.answers

In [ ]:
response.model

In [ ]:
print(f"Japan?  {response.nouls['is_japan'].noul:.1%}")
print(f"Scene:  {response.choices['scene'].choice} ({response.choices['scene'].confidence:.1%})")
print(f"Busy:   {response.scores['busy'].score:.2f} / 3")

## 1.3 Image Size

Jev only takes images with each side between **280 and 1568 px**, and it won't resize for you. This night photo is 1600 px wide, so it would be rejected.

A small helper that shrinks anything too big:

In [ ]:
MAX_SIDE = 1568


def load(path):
    img = PILImage.open(path).convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    return Image.from_pil(img)

In [ ]:
kabukicho = load("images/kabukicho.png")

## 1.4 Images Inside Questions

An `Image` can go anywhere content goes: in the state, in the instructions, even as `Choice` criteria.

Here the state is text and the options are pictures.

In [ ]:
temple = load("images/temple.png")

In [ ]:
questions = {
    "photo": Choice(
        instructions="Which photo matches the description?",
        criteria={
            "street": street,
            "kabukicho": kabukicho,
            "temple": temple,
        },
    ),
}

In [ ]:
for description in [
    "Neon signs on a rainy night",
    "Gold carvings of dragons",
    "Someone delivering pizza",
]:
    answer = client.system_one(state=description, questions=questions).choices["photo"]
    print(f"{description:<30} -> {answer.choice} ({answer.confidence:.1%})")

You can also put images inside a text state with an f-string. Attach them with `client.attach_assets` so the SDK knows to send them:

In [ ]:
with client.attach_assets([street, kabukicho]):
    response = client.system_one(
        state=f"Photo A: {street}\nPhoto B: {kabukicho}",
        questions={
            "same_city": Noul(instructions="Could these two photos be from the same city?"),
            "busier": Choice(
                instructions="Which photo shows the busier place?",
                criteria={"A": None, "B": None},
            ),
        },
    )

In [ ]:
response.answers

## 1.5 Your Screen as the State

Any screenshot is an image. Take one, ask Jev about it.

(macOS `screencapture`; your terminal needs Screen Recording permission. Retina screenshots are big, so `load` shrinks them.)

In [ ]:
import subprocess

subprocess.run(["screencapture", "-x", "-C", "screen.png"], check=True)
screen = load("screen.png")

In [ ]:
questions = {
    "activity": Choice(
        instructions="What is the person at this computer doing?",
        criteria={
            "coding": "An editor, terminal, or IDE with code",
            "writing": "Docs, notes, or email being written",
            "meeting": "A video call",
            "browsing": "Web pages, news, social media, or video",
        },
    ),
    "focused": Score(
        instructions="How focused on deep work does this screen look?",
        criteria=["Distracted", "Drifting", "Working", "Deep work"],
    ),
}

In [ ]:
response = client.system_one(state=screen, questions=questions)

In [ ]:
print(f"Activity: {response.choices['activity'].choice} ({response.choices['activity'].confidence:.1%})")
print(f"Focus:    {response.scores['focused'].score:.2f} / 3")

---
# 2. `Box`

## 2.1 Find Something in the Image

A `Box` question takes a description of what to find. The answer is the box's edges as **fractions** of the image (0 to 1), plus a confidence.

In [ ]:
questions = {
    "scooter": Box(instructions="The red delivery scooter"),
}

In [ ]:
response = client.system_one(state=street, questions=questions)

In [ ]:
response.boxes["scooter"]

Fractions are resolution-independent. Multiply by the image size to get pixels.

Let's write a small helper that draws boxes on the picture:

In [ ]:
def draw_boxes(path, boxes, max_width=900):
    img = PILImage.open(path).convert("RGB")
    draw = ImageDraw.Draw(img)
    w, h = img.size
    for name, answer in boxes.items():
        b = answer.box
        xy = (b.left * w, b.top * h, b.right * w, b.bottom * h)
        draw.rectangle(xy, outline="red", width=max(3, w // 200))
        draw.text((xy[0] + 6, xy[1] + 4), name, fill="red", font_size=max(16, w // 40))
    img.thumbnail((max_width, max_width))
    return img

In [ ]:
draw_boxes("images/street.png", response.boxes)

`Box` is the one feature that needs `jev-1.14.0`. Pin the old version and the same request is rejected with a bare `400 Invalid request`:

In [ ]:
client.system_one(state=street, questions=questions, model="jev-1.13.0")

## 2.2 Many Boxes in One Pass

Just like the first video: more questions, same single call. Here are four things to find in a busy night scene.

In [ ]:
questions = {
    "traffic_light": Box(instructions="The traffic light hanging over the street"),
    "don_quijote": Box(instructions="The Don Quijote store sign"),
    "black_car": Box(instructions="The black car driving past"),
    "seven_eleven": Box(instructions="The 7-Eleven logo"),
}

In [ ]:
response = client.system_one(state=kabukicho, questions=questions)

In [ ]:
for name, answer in response.boxes.items():
    print(f"{name:<14} confidence {answer.confidence:.1%}")

In [ ]:
draw_boxes("images/kabukicho.png", response.boxes)

## 2.3 Box + Noul: Only Trust the Box When the Thing Exists

`Box` always returns a box. If the object isn't there, you'll still get coordinates back.

So pair it with a `Noul` in the same pass, and use the box only when the answer is yes.

In [ ]:
questions = {
    "has_dog": Noul(instructions="Is there a dog in this photo?"),
    "dog": Box(instructions="The dog"),
    "has_umbrella": Noul(instructions="Is someone holding an umbrella?"),
    "umbrella": Box(instructions="The umbrella"),
}

In [ ]:
response = client.system_one(state=street, questions=questions)

In [ ]:
found = {}

for thing in ["dog", "umbrella"]:
    p = response.nouls[f"has_{thing}"].noul
    print(f"{thing:<9} exists: {p:.1%}   box confidence: {response.boxes[thing].confidence:.1%}")
    if p >= 0.8:
        found[thing] = response.boxes[thing]

In [ ]:
draw_boxes("images/street.png", found)

## 2.4 Box on Your Screen

Back to the screenshot from 1.5. Where's the terminal?

In [ ]:
response = client.system_one(state=screen, questions={"terminal": Box(instructions="The terminal window")})

In [ ]:
draw_boxes("screen.png", response.boxes)

That's the seed of the build: run this on a loop and act on the answers. We'll build it next (see the README).

---
# 3. Long Context

By default a request fits **64k tokens** (32k per question). The second opt-in header raises both to **262k**. It came with the same announcement, but it isn't a 1.14 feature: `jev-1.13.0` takes it too.

A real long document: Meta's quarterly report (10-Q) for Q3 2026, filed with the SEC on Oct 22, 2026. That's after Jev 1.14 came out, so Jev can't have seen it in training.

(Run the [appendix](#Appendix:-Download-the-Filing) once first to download it.)

In [ ]:
filing = open("texts/meta_10q_2026_q3.txt").read()
print(filing[:500])

In [ ]:
questions = {
    "revenue": Choice(
        instructions="How did revenue for the quarter compare with the same quarter last year?",
        criteria={"up more than 20%": None, "up less than 20%": None, "about flat": None, "down": None},
    ),
    "headcount": Choice(
        instructions="How did headcount change year over year?",
        criteria={"grew": None, "shrank": None, "about the same": None},
    ),
    "severance": Noul(instructions="Did Meta record severance costs this year?"),
    "tariffs": Noul(instructions="Does the filing list tariffs as a risk?"),
    "data_centers": Noul(instructions="Is Meta planning to dispose of any data center assets?"),
    "net_loss": Noul(instructions="Did Meta report a net loss for the quarter?"),
}

In [ ]:
client.system_one(state=filing, questions=questions)

Too long (`max_tokens_exceeded`). Now with the header:

In [ ]:
long_client = TypeSafeClient(headers={"TypeSafe-Danger-Zone-Long-Context": "true"}, timeout=60)

In [ ]:
import time

start = time.perf_counter()
response = long_client.system_one(state=filing, questions=questions)
print(f"{time.perf_counter() - start:.2f}s for {response.usage.input_tokens:,} tokens")

In [ ]:
for name, answer in response.choices.items():
    print(f"{name:<12} {answer.choice} ({answer.confidence:.1%})")
for name, answer in response.nouls.items():
    print(f"{name:<12} {answer.noul:.1%}")

Check them against the filing (Ctrl-F):

| Question | In the filing |
|---|---|
| `revenue` | $64.60B vs $51.24B a year earlier: up 26% |
| `headcount` | 66,775, "a decrease of 15% year-over-year" |
| `severance` | $1.18B of severance expenses in the nine months |
| `tariffs` | Listed among the risks to Meta's suppliers and products |
| `data_centers` | A July 2026 plan to dispose of $1.89B of data center assets |
| `net_loss` | Net income of $9.86B: no loss |

TypeSafe's warning: long context is slower and less accurate, so only turn it on for requests that need it.

---
# Appendix: Download the Filing

The filing isn't in the repo. This downloads it from SEC EDGAR, which is free with no API key. The SEC asks every client to send a `User-Agent` with a name and email: set `SEC_USER_AGENT` in `.env` (see `.env.example`).

Three requests: ticker → company ID (CIK) → its latest 10-Q → the document itself.

In [ ]:
import os

import httpx
from bs4 import BeautifulSoup

SEC_HEADERS = {"User-Agent": os.environ["SEC_USER_AGENT"]}

tickers = httpx.get("https://www.sec.gov/files/company_tickers.json", headers=SEC_HEADERS).json()
cik = next(t["cik_str"] for t in tickers.values() if t["ticker"] == "META")

recent = httpx.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json", headers=SEC_HEADERS).json()["filings"]["recent"]
i = recent["accessionNumber"].index("0001628280-26-066773")  # Meta's Q3 2026 10-Q
recent["form"][i], recent["filingDate"][i], recent["reportDate"][i]

In [ ]:
accession = recent["accessionNumber"][i].replace("-", "")
url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession}/{recent['primaryDocument'][i]}"
soup = BeautifulSoup(httpx.get(url, headers=SEC_HEADERS).text, "html.parser")

for hidden in soup.select('[style*="display:none"]'):  # machine-readable XBRL data, not the report text
    hidden.decompose()

Path("texts").mkdir(exist_ok=True)
path = Path("texts/meta_10q_2026_q3.txt")
path.write_text(soup.get_text("\n", strip=True))
print(f"Saved {path}")